In [4]:
import pandas as pd
import numpy as np
from rdkit import Chem
from rdkit.Chem import MACCSkeys, AllChem, Descriptors, Lipinski
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.metrics import roc_auc_score, accuracy_score

# Helper function to extract descriptors [cite: 59]
def extract_fingerprints(smiles):
    try:
        mol = Chem.MolFromSmiles(smiles)
        if mol is None: return None, None
        
        # 166-bit MACCS Keys
        maccs = MACCSkeys.GenMACCSKeys(mol)
        maccs_bits = np.array([int(b) for b in maccs.ToBitString()])
        
        # 1024-bit Morgan Fingerprint (Radius = 2 ~ Morgan3 descriptor equivalent)
        morgan = AllChem.GetMorganFingerprintAsBitVect(mol, radius=2, nBits=1024)
        morgan_bits = np.array([int(b) for b in morgan.ToBitString()])
        
        return maccs_bits, morgan_bits
    except:
        return None, None

# Helper function for Lipinski's Rule of Five [cite: 62, 94]
def calculate_lipinski_violations(smiles):
    try:
        mol = Chem.MolFromSmiles(smiles)
        if mol is None: return None, None, None, None, 4
        
        mw = Descriptors.MolWt(mol)
        logp = Descriptors.MolLogP(mol)
        h_donors = Lipinski.NumHDonors(mol)
        h_acceptors = Lipinski.NumHAcceptors(mol)
        
        violations = 0
        if mw > 500: violations += 1
        if logp > 5: violations += 1
        if h_donors > 5: violations += 1
        if h_acceptors > 10: violations += 1
        
        return mw, logp, h_donors, h_acceptors, violations
    except:
        return None, None, None, None, 4


In [5]:

# =====================================================================
# A. LOAD THE PREPARED DATASET
# =====================================================================
print("--- LOADING PREPARED CHEMBL TRAINING SET ---")
df_train_ready = pd.read_csv("../data/raw_data/HER2_ChEMBL_prepared_training_set.csv")

maccs_features, morgan_features, targets = [], [], []

print("Extracting MACCS and Morgan3 fingerprint bit vectors...")
for idx, row in df_train_ready.iterrows():
    maccs_fp, morgan_fp = extract_fingerprints(row['canonical_smiles'])
    if maccs_fp is not None:
        maccs_features.append(maccs_fp)
        morgan_features.append(morgan_fp)
        targets.append(row['binary_outcome'])

X_maccs = np.array(maccs_features)
X_morgan = np.array(morgan_features)
y = np.array(targets)


--- LOADING PREPARED CHEMBL TRAINING SET ---
Extracting MACCS and Morgan3 fingerprint bit vectors...


[23:06:14] DEPRECATION WARNING: please use MorganGenerator
[23:06:14] DEPRECATION WARNING: please use MorganGenerator
[23:06:14] DEPRECATION WARNING: please use MorganGenerator
[23:06:14] DEPRECATION WARNING: please use MorganGenerator
[23:06:14] DEPRECATION WARNING: please use MorganGenerator
[23:06:14] DEPRECATION WARNING: please use MorganGenerator
[23:06:14] DEPRECATION WARNING: please use MorganGenerator
[23:06:14] DEPRECATION WARNING: please use MorganGenerator
[23:06:14] DEPRECATION WARNING: please use MorganGenerator
[23:06:14] DEPRECATION WARNING: please use MorganGenerator
[23:06:14] DEPRECATION WARNING: please use MorganGenerator
[23:06:14] DEPRECATION WARNING: please use MorganGenerator
[23:06:14] DEPRECATION WARNING: please use MorganGenerator
[23:06:14] DEPRECATION WARNING: please use MorganGenerator
[23:06:14] DEPRECATION WARNING: please use MorganGenerator
[23:06:14] DEPRECATION WARNING: please use MorganGenerator
[23:06:14] DEPRECATION WARNING: please use MorganGenerat

In [6]:
# =====================================================================
# B. MODEL OPTIMIZATION VIA 5-FOLD CROSS-VALIDATION
# =====================================================================
print("\n--- TUNING ML HYPERPARAMETERS VIA GRID SEARCH ---")
rf_grid_params = {'n_estimators': [50, 100, 150], 'max_depth': [10, 20, None]}
svm_grid_params = {'C': [1, 10], 'kernel': ['rbf']}

fingerprints = {"MACCS (166-bit)": X_maccs, "Morgan3 (1024-bit)": X_morgan}

# FIX 1: Explicitly keep track of the absolute best configuration
best_auc = 0
winning_model_type = None     # Will store 'RF' or 'SVM'
winning_hyperparameters = None # Will store the best grid search params
winning_fp_name = ""          # Will store "MACCS (166-bit)" or "Morgan3 (1024-bit)"

for fp_name, X_data in fingerprints.items():
    # Stratified split ensures stable target balancing across training iterations
    X_train, X_val, y_train, y_val = train_test_split(X_data, y, test_size=0.20, random_state=42, stratify=y)
    
    # Tune Random Forest 
    rf_search = GridSearchCV(RandomForestClassifier(random_state=42), rf_grid_params, cv=5, scoring='roc_auc', n_jobs=-1)
    rf_search.fit(X_train, y_train)
    rf_auc = roc_auc_score(y_val, rf_search.best_estimator_.predict_proba(X_val)[:, 1])
    
    if rf_auc > best_auc:
        best_auc = rf_auc
        winning_model_type = 'RF'
        winning_hyperparameters = rf_search.best_params_
        winning_fp_name = fp_name
        
    # Tune SVM 
    svm_search = GridSearchCV(SVC(probability=True, random_state=42), svm_grid_params, cv=5, scoring='roc_auc', n_jobs=-1)
    svm_search.fit(X_train, y_train)
    svm_auc = roc_auc_score(y_val, svm_search.best_estimator_.predict_proba(X_val)[:, 1])
    
    if svm_auc > best_auc:
        best_auc = svm_auc
        winning_model_type = 'SVM'
        winning_hyperparameters = svm_search.best_params_
        winning_fp_name = fp_name

print(f"🏆 Winning Setup: {winning_model_type} utilizing {winning_fp_name} (Validation AUC: {best_auc:.3f})")


--- TUNING ML HYPERPARAMETERS VIA GRID SEARCH ---


d:\PriceTracker\ng\Lib\site-packages\sklearn\svm\_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
d:\PriceTracker\ng\Lib\site-packages\sklearn\svm\_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(


🏆 Winning Setup: RF utilizing Morgan3 (1024-bit) (Validation AUC: 0.931)


In [9]:
# =====================================================================
# B.5 RE-FIT CHAMPION MODEL ON COMPLETE DATASET (Crucial Missing Step)
# =====================================================================
print(f"\nRe-training the absolute best {winning_model_type} configuration on all available data...")
X_final_training_data = fingerprints[winning_fp_name]

if winning_model_type == 'RF':
    best_model = RandomForestClassifier(random_state=42, **winning_hyperparameters)
else:
    best_model = SVC(probability=True, random_state=42, **winning_hyperparameters)

# Train on 100% of the training dataset before screening
best_model.fit(X_final_training_data, y)


# =====================================================================
# C. VIRTUAL SCREENING OF THE PUBCHEM POOL (Optimized Speed Fix)
# =====================================================================
print("\n--- SCREENING THE PUBCHEM POOL ---")
df_pubchem = pd.read_csv("../data/raw_data/pubchem_erbb2_screening_set.csv")

# FIX 2: Vectorized high-speed mapping instead of slow row-by-row iterrows()
print("Extracting fingerprints from PubChem screening sets...")
fps_extracted = df_pubchem['ConnectivitySMILES'].apply(extract_fingerprints)

screen_features = []
valid_indices = []

# Collect only the specific fingerprint that won the competition step
for idx, (maccs_fp, morgan_fp) in enumerate(fps_extracted):
    if maccs_fp is not None:
        chosen_fp = maccs_fp if "MACCS" in winning_fp_name else morgan_fp
        screen_features.append(chosen_fp)
        valid_indices.append(idx)

# Align the original dataframe with successfully parsed chemical structures
df_results = df_pubchem.iloc[valid_indices].copy()
X_screen = np.array(screen_features)

# Predict outcomes using the fully matched model
df_results['ML_Prediction'] = best_model.predict(X_screen)

df_actives = df_results[df_results['ML_Prediction'] == 1].copy()
print(f"Total Predicted Actives passing ML classification: {len(df_actives)}")


Re-training the absolute best RF configuration on all available data...

--- SCREENING THE PUBCHEM POOL ---
Extracting fingerprints from PubChem screening sets...


[23:09:14] DEPRECATION WARNING: please use MorganGenerator
[23:09:14] DEPRECATION WARNING: please use MorganGenerator
[23:09:14] DEPRECATION WARNING: please use MorganGenerator
[23:09:14] DEPRECATION WARNING: please use MorganGenerator
[23:09:14] DEPRECATION WARNING: please use MorganGenerator
[23:09:14] DEPRECATION WARNING: please use MorganGenerator
[23:09:14] DEPRECATION WARNING: please use MorganGenerator
[23:09:14] DEPRECATION WARNING: please use MorganGenerator
[23:09:14] DEPRECATION WARNING: please use MorganGenerator
[23:09:14] DEPRECATION WARNING: please use MorganGenerator
[23:09:14] DEPRECATION WARNING: please use MorganGenerator
[23:09:14] DEPRECATION WARNING: please use MorganGenerator
[23:09:14] DEPRECATION WARNING: please use MorganGenerator
[23:09:14] DEPRECATION WARNING: please use MorganGenerator
[23:09:14] DEPRECATION WARNING: please use MorganGenerator
[23:09:14] DEPRECATION WARNING: please use MorganGenerator
[23:09:14] DEPRECATION WARNING: please use MorganGenerat

Total Predicted Actives passing ML classification: 3487


[23:09:22] DEPRECATION WARNING: please use MorganGenerator
[23:09:22] DEPRECATION WARNING: please use MorganGenerator
[23:09:22] DEPRECATION WARNING: please use MorganGenerator
[23:09:22] DEPRECATION WARNING: please use MorganGenerator
[23:09:22] DEPRECATION WARNING: please use MorganGenerator
[23:09:22] DEPRECATION WARNING: please use MorganGenerator
[23:09:22] DEPRECATION WARNING: please use MorganGenerator
[23:09:22] DEPRECATION WARNING: please use MorganGenerator
[23:09:22] DEPRECATION WARNING: please use MorganGenerator
[23:09:22] DEPRECATION WARNING: please use MorganGenerator
[23:09:22] DEPRECATION WARNING: please use MorganGenerator
[23:09:22] DEPRECATION WARNING: please use MorganGenerator
[23:09:22] DEPRECATION WARNING: please use MorganGenerator
[23:09:22] DEPRECATION WARNING: please use MorganGenerator
[23:09:22] DEPRECATION WARNING: please use MorganGenerator
[23:09:22] DEPRECATION WARNING: please use MorganGenerator
[23:09:22] DEPRECATION WARNING: please use MorganGenerat

In [11]:

# =====================================================================
# D. LIPINSKI'S RULE OF FIVE DRUG-LIKENESS FILTER [cite: 62, 94, 124]
# =====================================================================
print("\n--- APPLYING LIPINSKI DRUG-LIKENESS FILTER ---")
mw_l, logp_l, hd_l, ha_l, viol_l = [], [], [], [], []

for idx, row in df_actives.iterrows():
    mw, logp, hd, ha, viol = calculate_lipinski_violations(row['ConnectivitySMILES'])
    mw_l.append(mw); logp_l.append(logp); hd_l.append(hd); ha_l.append(ha); viol_l.append(viol)

df_actives['MW'] = mw_l; df_actives['LogP'] = logp_l; df_actives['H_Donors'] = hd_l
df_actives['H_Acceptors'] = ha_l; df_actives['Lipinski_Violations'] = viol_l

# Retain structures meeting the strict criteria (maximum 1 violation permitted) [cite: 124]
df_final_leads = df_actives[df_actives['Lipinski_Violations'] <= 1]

output_csv = "../data/raw_data/final_drug_like_her2_hits.csv"
df_final_leads.to_csv(output_csv, index=False)
print(f"\n🎉 Virtual Screening Complete! Saved {len(df_final_leads)} drug-like hits to '{output_csv}'.")


--- APPLYING LIPINSKI DRUG-LIKENESS FILTER ---

🎉 Virtual Screening Complete! Saved 2309 drug-like hits to '../data/raw_data/final_drug_like_her2_hits.csv'.


In [13]:
from sklearn.metrics import confusion_matrix # For computing sensitivity & specificity

# =====================================================================
# B. EXPLICIT MODEL EVALUATION MAPPED TO TABLE 1
# =====================================================================
print("\n--- REPLICATING TABLE 1 MODEL BENCHMARKING ---")

# Definition of the distinct combinations & exact hyperparameters from the paper
model_configurations = [
    {
        "name": "SVM-Morgan3",
        "features": X_morgan,
        "feature_name": "Morgan3 (1024-bit)",
        "estimator": SVC(kernel='rbf', gamma=10, C=10, probability=True, random_state=42)
    },
    {
        "name": "SVM-MACCS",
        "features": X_maccs,
        "feature_name": "MACCS (166-bit)",
        "estimator": SVC(kernel='poly', C=10, gamma=0.01, probability=True, random_state=42)
    },
    {
        "name": "RF-MACCS",
        "features": X_maccs,
        "feature_name": "MACCS (166-bit)",
        "estimator": RandomForestClassifier(n_estimators=150, criterion='entropy', max_depth=None, 
                                             min_samples_split=2, min_samples_leaf=1, random_state=42)
    },
    {
        "name": "RF-Morgan3",
        "features": X_morgan,
        "feature_name": "Morgan3 (1024-bit)",
        "estimator": RandomForestClassifier(n_estimators=50, criterion='entropy', max_depth=10, 
                                             min_samples_split=2, min_samples_leaf=4, random_state=42)
    }
]

performance_records = []
best_auc = 0
best_model = None
winning_fp_name = ""

for config in model_configurations:
    # Maintain consistent 80/20 stratified validation splitting
    X_train, X_val, y_train, y_val = train_test_split(
        config["features"], y, test_size=0.20, random_state=42, stratify=y
    )
    
    # Train configuration
    clf = config["estimator"]
    clf.fit(X_train, y_train)
    
    # Evaluate configurations
    y_pred = clf.predict(X_val)
    y_proba = clf.predict_proba(X_val)[:, 1]
    
    # Compute base evaluations
    accuracy = accuracy_score(y_val, y_pred)
    auc_score = roc_auc_score(y_val, y_proba)
    
    # Compute Sensitivity (Recall) and Specificity via Confusion Matrix
    # TN: True Inactives, FP: False Actives, FN: False Inactives, TP: True Actives
    tn, fp, fn, tp = confusion_matrix(y_val, y_pred).ravel()
    sensitivity = tp / (tp + fn) if (tp + fn) > 0 else 0
    specificity = tn / (tn + fp) if (tn + fp) > 0 else 0
    
    performance_records.append({
        "Model": config["name"],
        "Accuracy": accuracy,
        "Sensitivity": sensitivity,
        "Specificity": specificity,
        "AUC": auc_score
    })
    
    # Track champion framework for Section C virtual screening (RF-Morgan3 was the paper's final selection)
    if auc_score > best_auc:
        best_auc = auc_score
        best_model = clf
        winning_fp_name = config["feature_name"]

# Print the final formatted benchmarking summary table
df_table_1 = pd.DataFrame(performance_records)
print("\n================== BENCHMARK PERFORMANCE RESULTS ==================")
print(df_table_1.to_string(index=False, formatters={
    'Accuracy': '{:,.2f}'.format,
    'Sensitivity': '{:,.2f}'.format,
    'Specificity': '{:,.2f}'.format,
    'AUC': '{:,.2f}'.format
}))
print("===================================================================\n")

# =====================================================================
# B.5 RE-FIT CHAMPION MODEL ON 100% OF CORRESPONDING DATASET
# =====================================================================
print(f"Re-fitting champion architecture ({best_model.__class__.__name__}) on complete {winning_fp_name} feature pool...")
X_final_training_data = X_morgan if "Morgan3" in winning_fp_name else X_maccs
best_model.fit(X_final_training_data, y)


--- REPLICATING TABLE 1 MODEL BENCHMARKING ---


d:\PriceTracker\ng\Lib\site-packages\sklearn\svm\_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
d:\PriceTracker\ng\Lib\site-packages\sklearn\svm\_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(



================== BENCHMARK PERFORMANCE RESULTS ==================
      Model Accuracy Sensitivity Specificity  AUC
SVM-Morgan3     0.60        0.99        0.06 0.58
  SVM-MACCS     0.85        0.90        0.78 0.91
   RF-MACCS     0.84        0.89        0.77 0.91
 RF-Morgan3     0.83        0.91        0.72 0.91

Re-fitting champion architecture (SVC) on complete MACCS (166-bit) feature pool...


d:\PriceTracker\ng\Lib\site-packages\sklearn\svm\_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(


,"C C: float, default=1.0Regularization parameter. The strength of the regularization isinversely proportional to C. Must be strictly positive. The penaltyis a squared l2 penalty. For an intuitive visualization of the effectsof scaling the regularization parameter C, see:ref:`sphx_glr_auto_examples_svm_plot_svm_scale_c.py`.",10
,"kernel kernel: {'linear', 'poly', 'rbf', 'sigmoid', 'precomputed'} or callable, default='rbf'Specifies the kernel type to be used in the algorithm. Ifnone is given, 'rbf' will be used. If a callable is given it is used topre-compute the kernel matrix from data matrices; that matrix should bean array of shape ``(n_samples, n_samples)``. For an intuitivevisualization of different kernel types see:ref:`sphx_glr_auto_examples_svm_plot_svm_kernels.py`.",'poly'
,"gamma gamma: {'scale', 'auto'} or float, default='scale'Kernel coefficient for 'rbf', 'poly' and 'sigmoid'.- if ``gamma='scale'`` (default) is passed then it uses 1 / (n_features * X.var()) as value of gamma,- if 'auto', uses 1 / n_features- if float, must be non-negative... versionchanged:: 0.22 The default value of ``gamma`` changed from 'auto' to 'scale'.",0.01
,"probability probability: bool, default=FalseWhether to enable probability estimates. This must be enabled priorto calling `fit`, will slow down that method as it internally uses5-fold cross-validation, and `predict_proba` may be inconsistent with`predict`. Read more in the :ref:`User Guide <scores_probabilities>`...deprecated:: 1.9 The `probability` parameter is deprecated and will be removed in 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`.",True
,"random_state random_state: int, RandomState instance or None, default=NoneControls the pseudo random number generation for shuffling the data forprobability estimates. Ignored when `probability` is False.Pass an int for reproducible output across multiple function calls.See :term:`Glossary <random_state>`.",42
,"degree degree: int, default=3Degree of the polynomial kernel function ('poly').Must be non-negative. Ignored by all other kernels.",3
,"coef0 coef0: float, default=0.0Independent term in kernel function.It is only significant in 'poly' and 'sigmoid'.",0.0
,"shrinking shrinking: bool, default=TrueWhether to use the shrinking heuristic.See the :ref:`User Guide <shrinking_svm>`.",True
,"tol tol: float, default=1e-3Tolerance for stopping criterion.",0.001
,"cache_size cache_size: float, default=200Specify the size of the kernel cache (in MB).",200
,"class_weight class_weight: dict or 'balanced', default=NoneSet the parameter C of class i to class_weight[i]*C forSVC. If not given, all classes are supposed to haveweight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.",None


In [ ]:
from sklearn.metrics import confusion_matrix

# =====================================================================
# B. HYPERPARAMETER TUNING & FINE-TUNING VIA GRID SEARCH (CUSTOM FOR HER2)
# =====================================================================
print("\n--- FINE-TUNING MODEL HYPERPARAMETERS VIA GRID SEARCH ---")

# Define hyperparameter grids to explore for your HER2 dataset
rf_grid_params = {
    'n_estimators': [50, 100, 150, 200],
    'max_depth': [10, 20, 30, None],
    'criterion': ['gini', 'entropy'],
    'min_samples_split': [2, 5],
    'min_samples_leaf': [1, 2, 4]
}

svm_grid_params = {
    'C': [0.1, 1, 10, 100],
    'gamma': [0.01, 0.1, 1, 10, 'scale'],
    'kernel': ['rbf', 'poly']
}

fingerprints = {"MACCS (166-bit)": X_maccs, "Morgan3 (1024-bit)": X_morgan}

performance_records = []
best_auc = 0
winning_model_blueprint = None
winning_hyperparameters = None
winning_fp_name = ""
winning_model_type = ""

# Loop through both fingerprint types
for fp_name, X_data in fingerprints.items():
    print(f"Fine-tuning models using {fp_name} features...")
    
    # 80/20 Stratified Split for local validation evaluation
    X_train, X_val, y_train, y_val = train_test_split(
        X_data, y, test_size=0.20, random_state=42, stratify=y
    )
    
    # --- 1. Fine-Tune Random Forest ---
    rf_grid = GridSearchCV(
        RandomForestClassifier(random_state=42), 
        rf_grid_params, cv=5, scoring='roc_auc', n_jobs=-1
    )
    rf_grid.fit(X_train, y_train)
    
    # Evaluate Best RF Model on validation set
    best_rf = rf_grid.best_estimator_
    rf_pred = best_rf.predict(X_val)
    rf_proba = best_rf.predict_proba(X_val)[:, 1]
    
    # Calculate Metrics
    rf_acc = accuracy_score(y_val, rf_pred)
    rf_auc = roc_auc_score(y_val, rf_proba)
    tn, fp, fn, tp = confusion_matrix(y_val, rf_pred).ravel()
    rf_sens = tp / (tp + fn) if (tp + fn) > 0 else 0
    rf_spec = tn / (tn + fp) if (tn + fp) > 0 else 0
    
    performance_records.append({
        "Model Setup": f"RF + {fp_name.split()[0]}",
        "Best Params": str(rf_grid.best_params_),
        "Accuracy": rf_acc, "Sensitivity": rf_sens, "Specificity": rf_spec, "AUC": rf_auc
    })
    
    if rf_auc > best_auc:
        best_auc = rf_auc
        winning_model_type = "RF"
        winning_hyperparameters = rf_grid.best_params_
        winning_fp_name = fp_name

    # --- 2. Fine-Tune Support Vector Machine (SVM) ---
    svm_grid = GridSearchCV(
        SVC(probability=True, random_state=42), 
        svm_grid_params, cv=5, scoring='roc_auc', n_jobs=-1
    )
    svm_grid.fit(X_train, y_train)
    
    # Evaluate Best SVM Model on validation set
    best_svm = svm_grid.best_estimator_
    svm_pred = best_svm.predict(X_val)
    svm_proba = best_svm.predict_proba(X_val)[:, 1]
    
    # Calculate Metrics
    svm_acc = accuracy_score(y_val, svm_pred)
    svm_auc = roc_auc_score(y_val, svm_proba)
    tn, fp, fn, tp = confusion_matrix(y_val, svm_pred).ravel()
    svm_sens = tp / (tp + fn) if (tp + fn) > 0 else 0
    svm_spec = tn / (tn + fp) if (tn + fp) > 0 else 0
    
    performance_records.append({
        "Model Setup": f"SVM + {fp_name.split()[0]}",
        "Best Params": str(svm_grid.best_params_),
        "Accuracy": svm_acc, "Sensitivity": svm_sens, "Specificity": svm_spec, "AUC": svm_auc
    })
    
    if svm_auc > best_auc:
        best_auc = svm_auc
        winning_model_type = "SVM"
        winning_hyperparameters = svm_grid.best_params_
        winning_fp_name = fp_name

# =====================================================================
# DISPLAY YOUR CUSTOM "TABLE 1" FOR HER2
# =====================================================================
df_table_1 = pd.DataFrame(performance_records)
print("\n==================== CUSTOM HER2 BENCHMARK RESULTS ====================")
for idx, row in df_table_1.iterrows():
    print(f"\nModel Framework: {row['Model Setup']}")
    print(f"Optimal Fine-Tuned Hyperparameters: {row['Best Params']}")
    print(f"Performance -> Accuracy: {row['Accuracy']:.2f} | Sensitivity: {row['Sensitivity']:.2f} | Specificity: {row['Specificity']:.2f} | AUC: {row['AUC']:.2f}")
print("========================================================================\n")

print(f"🏆 Champion System Selected: {winning_model_type} running on {winning_fp_name}")

# =====================================================================
# B.5 RE-FIT THE CHAMPION SYSTEM ON 100% OF YOUR DATA
# =====================================================================
print(f"Final Step: Training the champion {winning_model_type} architecture on all available training data...")
X_final_pool = fingerprints[winning_fp_name]

if winning_model_type == "RF":
    best_model = RandomForestClassifier(random_state=42, **winning_hyperparameters)
else:
    best_model = SVC(probability=True, random_state=42, **winning_hyperparameters)

# Retrain model on all data to maximize learning performance before the PubChem screen
best_model.fit(X_final_pool, y)




# ==================== CUSTOM HER2 BENCHMARK RESULTS ====================

# Model Framework: RF + MACCS
# Optimal Fine-Tuned Hyperparameters: {'criterion': 'gini', 'max_depth': None, 'min_samples_leaf': 1, 'min_samples_split': 5, 'n_estimators': 200}
# Performance -> Accuracy: 0.85 | Sensitivity: 0.90 | Specificity: 0.77 | AUC: 0.92

# Model Framework: SVM + MACCS
# Optimal Fine-Tuned Hyperparameters: {'C': 1, 'gamma': 0.1, 'kernel': 'rbf'}
# Performance -> Accuracy: 0.85 | Sensitivity: 0.90 | Specificity: 0.78 | AUC: 0.92

# Model Framework: RF + Morgan3
# Optimal Fine-Tuned Hyperparameters: {'criterion': 'entropy', 'max_depth': 30, 'min_samples_leaf': 1, 'min_samples_split': 5, 'n_estimators': 200}
# Performance -> Accuracy: 0.86 | Sensitivity: 0.90 | Specificity: 0.81 | AUC: 0.93

# Model Framework: SVM + Morgan3
# Optimal Fine-Tuned Hyperparameters: {'C': 10, 'gamma': 0.01, 'kernel': 'poly'}
# Performance -> Accuracy: 0.87 | Sensitivity: 0.89 | Specificity: 0.84 | AUC: 0.93
# ========================================================================

# 🏆 Champion System Selected: RF running on Morgan3 (1024-bit)
# Final Step: Training the champion RF architecture on all available training data...


--- FINE-TUNING MODEL HYPERPARAMETERS VIA GRID SEARCH ---
Fine-tuning models using MACCS (166-bit) features...


d:\PriceTracker\ng\Lib\site-packages\sklearn\svm\_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(


Fine-tuning models using Morgan3 (1024-bit) features...


d:\PriceTracker\ng\Lib\site-packages\sklearn\svm\_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(



==================== CUSTOM HER2 BENCHMARK RESULTS ====================

Model Framework: RF + MACCS
Optimal Fine-Tuned Hyperparameters: {'criterion': 'gini', 'max_depth': None, 'min_samples_leaf': 1, 'min_samples_split': 5, 'n_estimators': 200}
Performance -> Accuracy: 0.85 | Sensitivity: 0.90 | Specificity: 0.77 | AUC: 0.92

Model Framework: SVM + MACCS
Optimal Fine-Tuned Hyperparameters: {'C': 1, 'gamma': 0.1, 'kernel': 'rbf'}
Performance -> Accuracy: 0.85 | Sensitivity: 0.90 | Specificity: 0.78 | AUC: 0.92

Model Framework: RF + Morgan3
Optimal Fine-Tuned Hyperparameters: {'criterion': 'entropy', 'max_depth': 30, 'min_samples_leaf': 1, 'min_samples_split': 5, 'n_estimators': 200}
Performance -> Accuracy: 0.86 | Sensitivity: 0.90 | Specificity: 0.81 | AUC: 0.93

Model Framework: SVM + Morgan3
Optimal Fine-Tuned Hyperparameters: {'C': 10, 'gamma': 0.01, 'kernel': 'poly'}
Performance -> Accuracy: 0.87 | Sensitivity: 0.89 | Specificity: 0.84 | AUC: 0.93

🏆 Champion System Selected: RF

,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",200
,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.Note: This parameter is tree-specific.",'entropy'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",30
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",5
,"random_state random_state: int, RandomState instance or None, default=NoneControls both the randomness of the bootstrapping of the samples usedwhen building trees (if ``bootstrap=True``) and the sampling of thefeatures to consider when looking for the best split at each node(if ``max_features < n_features``).See :term:`Glossary <random_state>` for details.",42
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=""sqrt""The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to `""sqrt""`.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",'sqrt'
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow trees with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsamples at the current node, ``N_t_L`` is the number of samples in theleft child, and ``N_t_R`` is the number of samples in the right child.``N``, ``N_t``, ``N_t_R`` and ``N_t_L`` all refer to the weighted sum,if ``sample_weight`` is passed... versionadded:: 0.19",0.0
,"bootstrap bootst

In [ ]:
# import joblib

# # 1. Save the final trained model
# model_filename = f"her2_champion_{winning_model_type.lower()}_model.joblib"
# joblib.dump(best_model, model_filename)
# print(f"💾 Model saved to: {model_filename}")

# # 2. Save the fingerprint metadata (tells you if it expects MACCS or Morgan3)
# metadata = {"winning_fingerprint_type": winning_fp_name}
# joblib.dump(metadata, "model_metadata.joblib")
# print("💾 Metadata saved to: model_metadata.joblib")
